# DeepSpeed ZeRO: Zero Redundancy Optimizer

**Historical Context**: ZeRO (Zero Redundancy Optimizer) was introduced by Microsoft in 2020 to enable training massive models that don't fit on a single GPU. It addresses the memory redundancy problem where each GPU stores identical copies of optimizer states, gradients, and parameters.

**Key Innovation**: By partitioning optimizer states, gradients, and parameters across GPUs, ZeRO eliminates memory redundancy while maintaining computational efficiency. This enables training models 100x larger than what fits on a single GPU.

**Learning Objectives**:
- Understand the ZeRO paper and motivation (2020)
- Learn optimizer state partitioning fundamentals
- Master ZeRO Stage 1, 2, and 3 configurations
- Analyze memory breakdown for each stage
- Implement training with DeepSpeed
- Scale experiments from 1 to 8 GPUs
- Know when to use each ZeRO stage

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import numpy as np
from typing import Dict, List, Tuple
import json

# Check if DeepSpeed is available
try:
    import deepspeed
    DEEPSPEED_AVAILABLE = True
    print(f"DeepSpeed version: {deepspeed.__version__}")
except ImportError:
    DEEPSPEED_AVAILABLE = False
    print("DeepSpeed not installed. Install with: pip install deepspeed")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Number of GPUs: {torch.cuda.device_count()}")

## 1. The Memory Redundancy Problem

**Traditional Data Parallel Training**:
- Each GPU holds a complete copy of:
  - Model parameters (P)
  - Gradients (G)
  - Optimizer states (OS): momentum, variance for Adam

**Memory Breakdown for Mixed Precision (FP16) Training**:

For a model with P parameters using Adam optimizer:

1. **Model Parameters**: 
   - FP16 copy: 2P bytes
   - FP32 master copy: 4P bytes
   - Total: 6P bytes

2. **Gradients**: 2P bytes (FP16)

3. **Optimizer States** (Adam):
   - Momentum: 4P bytes (FP32)
   - Variance: 4P bytes (FP32)
   - Total: 8P bytes

4. **Activations**: Depends on sequence length and batch size

**Total Memory per GPU**: 16P + activations

**Problem**: For 8 GPUs, 8x redundancy = wasted 7/8 of memory!

In [ ]:
def calculate_model_memory(num_parameters: int, num_gpus: int = 1, precision: str = 'fp16') -> Dict[str, float]:
    """
    Calculate memory requirements for training a model.
    
    Args:
        num_parameters: Number of model parameters
        num_gpus: Number of GPUs for data parallel training
        precision: 'fp16' or 'fp32'
    
    Returns:
        Dictionary with memory breakdown in GB
    """
    bytes_per_param = 2 if precision == 'fp16' else 4
    
    # Model parameters (mixed precision: FP16 + FP32 master)
    model_fp16 = (num_parameters * 2) / (1024**3)  # GB
    model_fp32 = (num_parameters * 4) / (1024**3)  # GB
    total_model = model_fp16 + model_fp32
    
    # Gradients (same precision as model)
    gradients = (num_parameters * bytes_per_param) / (1024**3)
    
    # Optimizer states (Adam: momentum + variance, both FP32)
    optimizer_states = (num_parameters * 8) / (1024**3)  # 2 states × 4 bytes
    
    # Total per GPU
    total_per_gpu = total_model + gradients + optimizer_states
    
    # Total across all GPUs (redundant)
    total_all_gpus = total_per_gpu * num_gpus
    
    return {
        'model_fp16': model_fp16,
        'model_fp32': model_fp32,
        'total_model': total_model,
        'gradients': gradients,
        'optimizer_states': optimizer_states,
        'total_per_gpu': total_per_gpu,
        'total_all_gpus': total_all_gpus,
        'redundant_memory': total_all_gpus - total_per_gpu,
    }

# Example: Calculate memory for different model sizes
model_sizes = {
    'GPT-2 (124M)': 124_000_000,
    'GPT-2 Large (774M)': 774_000_000,
    'GPT-3 Small (1.3B)': 1_300_000_000,
    'GPT-3 (6.7B)': 6_700_000_000,
    'GPT-3 (13B)': 13_000_000_000,
}

print("Memory Requirements for Different Models (8 GPUs, Mixed Precision):")
print("="*90)
print(f"{'Model':<20} {'Model':<10} {'Gradients':<10} {'Optimizer':<10} {'Per GPU':<10} {'Redundant':<10}")
print(f"{'':20} {'(GB)':<10} {'(GB)':<10} {'(GB)':<10} {'(GB)':<10} {'(GB)':<10}")
print("="*90)

for model_name, num_params in model_sizes.items():
    memory = calculate_model_memory(num_params, num_gpus=8)
    print(f"{model_name:<20} {memory['total_model']:<10.2f} {memory['gradients']:<10.2f} "
          f"{memory['optimizer_states']:<10.2f} {memory['total_per_gpu']:<10.2f} "
          f"{memory['redundant_memory']:<10.2f}")

print("\nNote: This does not include activation memory, which scales with batch size!")

## 2. ZeRO: Eliminating Memory Redundancy

**ZeRO Philosophy**: Each GPU should only store 1/N of optimizer states, gradients, and parameters.

**Three Stages of Optimization**:

### ZeRO Stage 1: Optimizer State Partitioning
- **What**: Partition optimizer states across GPUs
- **Memory Saving**: 4x reduction in optimizer memory
- **Communication**: None extra (same as data parallel)
- **Use Case**: Models that fit with this optimization

### ZeRO Stage 2: + Gradient Partitioning
- **What**: Partition optimizer states AND gradients
- **Memory Saving**: 8x reduction total
- **Communication**: Same as data parallel
- **Use Case**: Larger models, most common stage

### ZeRO Stage 3: + Parameter Partitioning
- **What**: Partition everything (optimizer, gradients, parameters)
- **Memory Saving**: Linear with number of GPUs
- **Communication**: Increased (gather parameters during forward/backward)
- **Use Case**: Very large models (10B+ parameters)

**Memory Formula**:
```
Stage 1: Memory = P × (6 + 2 + 8/N)
Stage 2: Memory = P × (6 + 2/N + 8/N) 
Stage 3: Memory = P × (6/N + 2/N + 8/N) = 16P/N
```
where N = number of GPUs, P = parameter count in bytes

In [ ]:
def calculate_zero_memory(num_parameters: int, num_gpus: int, zero_stage: int) -> Dict[str, float]:
    """
    Calculate memory with ZeRO optimization.
    
    Args:
        num_parameters: Number of model parameters
        num_gpus: Number of GPUs
        zero_stage: 0 (baseline), 1, 2, or 3
    """
    P = num_parameters / (1024**3)  # Convert to GB (4 bytes per param for base unit)
    
    if zero_stage == 0:  # Baseline data parallel
        model_memory = 6 * P
        gradient_memory = 2 * P
        optimizer_memory = 8 * P
    elif zero_stage == 1:  # Optimizer state partitioning
        model_memory = 6 * P
        gradient_memory = 2 * P
        optimizer_memory = (8 * P) / num_gpus
    elif zero_stage == 2:  # + Gradient partitioning
        model_memory = 6 * P
        gradient_memory = (2 * P) / num_gpus
        optimizer_memory = (8 * P) / num_gpus
    elif zero_stage == 3:  # + Parameter partitioning
        model_memory = (6 * P) / num_gpus
        gradient_memory = (2 * P) / num_gpus
        optimizer_memory = (8 * P) / num_gpus
    else:
        raise ValueError(f"Invalid zero_stage: {zero_stage}")
    
    total_per_gpu = model_memory + gradient_memory + optimizer_memory
    
    return {
        'model_memory': model_memory,
        'gradient_memory': gradient_memory,
        'optimizer_memory': optimizer_memory,
        'total_per_gpu': total_per_gpu,
    }

# Visualize memory savings
def visualize_zero_memory_savings():
    num_params = 1_300_000_000  # 1.3B parameter model
    gpu_counts = [1, 2, 4, 8]
    stages = [0, 1, 2, 3]
    stage_names = ['Baseline\n(Data Parallel)', 'ZeRO Stage 1\n(Optimizer)', 
                   'ZeRO Stage 2\n(+Gradients)', 'ZeRO Stage 3\n(+Parameters)']
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    axes = axes.flatten()
    
    for idx, num_gpus in enumerate(gpu_counts):
        ax = axes[idx]
        
        memories = []
        for stage in stages:
            mem = calculate_zero_memory(num_params, num_gpus, stage)
            memories.append([
                mem['model_memory'],
                mem['gradient_memory'],
                mem['optimizer_memory']
            ])
        
        memories = np.array(memories)
        
        x = np.arange(len(stages))
        width = 0.6
        
        # Stacked bar chart
        p1 = ax.bar(x, memories[:, 0], width, label='Model', alpha=0.8, color='#FF6B6B')
        p2 = ax.bar(x, memories[:, 1], width, bottom=memories[:, 0], 
                   label='Gradients', alpha=0.8, color='#4ECDC4')
        p3 = ax.bar(x, memories[:, 2], width, 
                   bottom=memories[:, 0] + memories[:, 1],
                   label='Optimizer', alpha=0.8, color='#45B7D1')
        
        ax.set_ylabel('Memory per GPU (GB)', fontsize=10)
        ax.set_title(f'{num_gpus} GPU(s) - 1.3B Parameters', fontsize=11, fontweight='bold')
        ax.set_xticks(x)
        ax.set_xticklabels(stage_names, fontsize=8)
        ax.legend(fontsize=9)
        ax.grid(axis='y', alpha=0.3)
        
        # Add total memory labels
        for i, total in enumerate(memories.sum(axis=1)):
            ax.text(i, total + 0.5, f'{total:.1f} GB', 
                   ha='center', va='bottom', fontsize=9, fontweight='bold')
    
    plt.tight_layout()
    plt.savefig('/Users/shyam.mukherjee/Desktop/research_and_exploration/os_ft/notebooks/zero_memory_savings.png', 
                dpi=150, bbox_inches='tight')
    plt.show()

visualize_zero_memory_savings()

In [ ]:
# Detailed comparison table
import pandas as pd

def create_zero_comparison_table():
    num_params = 1_300_000_000
    gpu_counts = [1, 2, 4, 8]
    
    data = []
    for num_gpus in gpu_counts:
        baseline = calculate_zero_memory(num_params, num_gpus, 0)
        for stage in [0, 1, 2, 3]:
            mem = calculate_zero_memory(num_params, num_gpus, stage)
            reduction = baseline['total_per_gpu'] / mem['total_per_gpu']
            data.append({
                'GPUs': num_gpus,
                'Stage': f"Stage {stage}" if stage > 0 else "Baseline",
                'Model (GB)': f"{mem['model_memory']:.2f}",
                'Gradients (GB)': f"{mem['gradient_memory']:.2f}",
                'Optimizer (GB)': f"{mem['optimizer_memory']:.2f}",
                'Total (GB)': f"{mem['total_per_gpu']:.2f}",
                'Reduction': f"{reduction:.2f}x",
            })
    
    df = pd.DataFrame(data)
    print("\nZeRO Memory Comparison (1.3B Parameters):")
    print("="*100)
    print(df.to_string(index=False))
    print("="*100)

create_zero_comparison_table()

## 3. How ZeRO Works: Under the Hood

### Stage 1: Optimizer State Partitioning

**Process**:
1. Forward pass: Each GPU computes on full model
2. Backward pass: Each GPU computes gradients
3. **AllReduce**: Average gradients across GPUs (same as data parallel)
4. **Update**: Each GPU updates only its assigned parameters
   - GPU 0 updates parameters 0-N/8
   - GPU 1 updates parameters N/8-2N/8
   - etc.

**Memory Saved**: Optimizer states (8P bytes) → (8P/N bytes)

### Stage 2: + Gradient Partitioning

**Process**:
1. Forward pass: Same as Stage 1
2. Backward pass: Each GPU computes gradients
3. **Reduce-Scatter**: Instead of AllReduce, each GPU receives average for its partition
4. **Update**: Each GPU updates its partition

**Memory Saved**: Gradients (2P) + Optimizer (8P) → (10P/N bytes)

### Stage 3: + Parameter Partitioning

**Process**:
1. **Forward pass**: 
   - Before each layer: **AllGather** to collect full parameters
   - Compute layer
   - Release non-owned parameters
2. **Backward pass**: Similar gathering for gradients
3. **Update**: Each GPU updates its partition

**Memory Saved**: Everything partitioned → 16P/N bytes

**Trade-off**: More communication (AllGather) but massive memory savings

In [ ]:
def visualize_zero_communication_patterns():
    """
    Visualize communication patterns for different ZeRO stages.
    """
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    stages_info = [
        {
            'title': 'Baseline Data Parallel',
            'operations': ['Forward:\nLocal compute', 'Backward:\nLocal gradients', 
                         'AllReduce:\nAverage gradients', 'Update:\nLocal optimizer'],
            'communication': [0, 0, 1, 0],
        },
        {
            'title': 'ZeRO Stage 1',
            'operations': ['Forward:\nLocal compute', 'Backward:\nLocal gradients',
                         'AllReduce:\nAverage gradients', 'Update:\nPartitioned'],
            'communication': [0, 0, 1, 0],
        },
        {
            'title': 'ZeRO Stage 2',
            'operations': ['Forward:\nLocal compute', 'Backward:\nLocal gradients',
                         'Reduce-Scatter:\nPartition gradients', 'Update:\nPartitioned'],
            'communication': [0, 0, 1, 0],
        },
        {
            'title': 'ZeRO Stage 3',
            'operations': ['AllGather:\nCollect params', 'Forward:\nCompute',
                         'AllGather+Backward:\nGradients', 'Update:\nPartitioned'],
            'communication': [1, 0, 1, 0],
        },
    ]
    
    for idx, (ax, stage_info) in enumerate(zip(axes.flatten(), stages_info)):
        operations = stage_info['operations']
        communication = stage_info['communication']
        
        x = np.arange(len(operations))
        colors = ['#45B7D1' if comm else '#95E1D3' for comm in communication]
        
        bars = ax.bar(x, [1]*len(operations), color=colors, alpha=0.8, edgecolor='black', linewidth=2)
        
        # Add labels on bars
        for i, (op, comm) in enumerate(zip(operations, communication)):
            ax.text(i, 0.5, op, ha='center', va='center', fontsize=9, fontweight='bold')
            if comm:
                ax.text(i, 1.1, 'Communication', ha='center', fontsize=8, 
                       color='red', fontweight='bold')
        
        ax.set_ylim(0, 1.3)
        ax.set_xlim(-0.5, len(operations)-0.5)
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_title(stage_info['title'], fontsize=12, fontweight='bold', pad=10)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.spines['left'].set_visible(False)
        ax.spines['bottom'].set_visible(False)
    
    plt.tight_layout()
    plt.savefig('/Users/shyam.mukherjee/Desktop/research_and_exploration/os_ft/notebooks/zero_communication.png',
                dpi=150, bbox_inches='tight')
    plt.show()

visualize_zero_communication_patterns()

## 4. DeepSpeed Configuration Files

DeepSpeed uses JSON configuration files to specify ZeRO settings:

In [ ]:
# ZeRO Stage 1 Configuration
zero_stage1_config = {
    "train_batch_size": 32,
    "gradient_accumulation_steps": 1,
    "fp16": {
        "enabled": True
    },
    "zero_optimization": {
        "stage": 1,
        "reduce_bucket_size": 5e8,
        "allgather_bucket_size": 5e8
    },
    "optimizer": {
        "type": "AdamW",
        "params": {
            "lr": 3e-5,
            "betas": [0.9, 0.999],
            "eps": 1e-8,
            "weight_decay": 0.01
        }
    }
}

# ZeRO Stage 2 Configuration
zero_stage2_config = {
    "train_batch_size": 32,
    "gradient_accumulation_steps": 1,
    "fp16": {
        "enabled": True
    },
    "zero_optimization": {
        "stage": 2,
        "offload_optimizer": {
            "device": "cpu",
            "pin_memory": True
        },
        "allgather_partitions": True,
        "allgather_bucket_size": 5e8,
        "reduce_scatter": True,
        "reduce_bucket_size": 5e8,
        "overlap_comm": True,
        "contiguous_gradients": True
    },
    "optimizer": {
        "type": "AdamW",
        "params": {
            "lr": 3e-5,
            "betas": [0.9, 0.999],
            "eps": 1e-8,
            "weight_decay": 0.01
        }
    }
}

# ZeRO Stage 3 Configuration
zero_stage3_config = {
    "train_batch_size": 32,
    "gradient_accumulation_steps": 1,
    "fp16": {
        "enabled": True
    },
    "zero_optimization": {
        "stage": 3,
        "offload_optimizer": {
            "device": "cpu",
            "pin_memory": True
        },
        "offload_param": {
            "device": "cpu",
            "pin_memory": True
        },
        "overlap_comm": True,
        "contiguous_gradients": True,
        "sub_group_size": 1e9,
        "reduce_bucket_size": 5e8,
        "stage3_prefetch_bucket_size": 5e8,
        "stage3_param_persistence_threshold": 1e6,
        "stage3_max_live_parameters": 1e9,
        "stage3_max_reuse_distance": 1e9,
        "stage3_gather_16bit_weights_on_model_save": True
    },
    "optimizer": {
        "type": "AdamW",
        "params": {
            "lr": 3e-5,
            "betas": [0.9, 0.999],
            "eps": 1e-8,
            "weight_decay": 0.01
        }
    }
}

# Save configs
configs = {
    'zero_stage1': zero_stage1_config,
    'zero_stage2': zero_stage2_config,
    'zero_stage3': zero_stage3_config,
}

for name, config in configs.items():
    with open(f'/Users/shyam.mukherjee/Desktop/research_and_exploration/os_ft/notebooks/{name}_config.json', 'w') as f:
        json.dump(config, f, indent=2)
    print(f"Saved: {name}_config.json")

print("\nKey Configuration Parameters:")
print("="*70)
print("Stage 1: Only optimizer state partitioning")
print("Stage 2: + Gradient partitioning + CPU offloading")
print("Stage 3: + Parameter partitioning + CPU offloading")
print("\nCPU Offloading: Move optimizer/parameters to CPU to save GPU memory")
print("Overlap Communication: Hide communication behind computation")

## 5. Implementation: Training with DeepSpeed

Here's how to train a model with DeepSpeed ZeRO:

In [ ]:
# Simple transformer model for demonstration
class SimpleTransformer(nn.Module):
    def __init__(self, vocab_size=10000, d_model=512, nhead=8, num_layers=6):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=d_model*4,
            dropout=0.1, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.fc_out = nn.Linear(d_model, vocab_size)
        
    def forward(self, x):
        x = self.embedding(x)
        x = self.transformer(x)
        x = self.fc_out(x)
        return x

# Calculate model size
model = SimpleTransformer()
num_params = sum(p.numel() for p in model.parameters())
print(f"Model size: {num_params:,} parameters ({num_params/1e6:.2f}M)")
print(f"Model memory: {num_params * 4 / (1024**2):.2f} MB (FP32)")

# Training script (conceptual - requires multi-GPU setup)
training_script = '''
# train_with_deepspeed.py
import torch
import deepspeed
from torch.utils.data import DataLoader, Dataset
import argparse

class DummyDataset(Dataset):
    def __init__(self, num_samples=1000, seq_len=512, vocab_size=10000):
        self.num_samples = num_samples
        self.seq_len = seq_len
        self.vocab_size = vocab_size
    
    def __len__(self):
        return self.num_samples
    
    def __getitem__(self, idx):
        return torch.randint(0, self.vocab_size, (self.seq_len,))

def main():
    # Parse arguments
    parser = argparse.ArgumentParser()
    parser.add_argument('--local_rank', type=int, default=-1)
    parser.add_argument('--deepspeed_config', type=str, required=True)
    args = parser.parse_args()
    
    # Create model
    model = SimpleTransformer()
    
    # Create dataset
    dataset = DummyDataset()
    
    # Initialize DeepSpeed
    model_engine, optimizer, dataloader, _ = deepspeed.initialize(
        model=model,
        model_parameters=model.parameters(),
        training_data=dataset,
        config=args.deepspeed_config
    )
    
    # Training loop
    for epoch in range(3):
        for batch_idx, batch in enumerate(dataloader):
            # Forward pass
            outputs = model_engine(batch)
            
            # Compute loss (simple example)
            loss = outputs.mean()
            
            # Backward pass (handled by DeepSpeed)
            model_engine.backward(loss)
            
            # Optimizer step (handled by DeepSpeed)
            model_engine.step()
            
            if batch_idx % 10 == 0:
                print(f"Epoch {epoch}, Batch {batch_idx}, Loss: {loss.item():.4f}")
    
    # Save checkpoint
    model_engine.save_checkpoint('checkpoints', tag='final')

if __name__ == '__main__':
    main()

# Run with:
# deepspeed --num_gpus=8 train_with_deepspeed.py --deepspeed_config zero_stage2_config.json
'''

print("\nDeepSpeed Training Script:")
print("="*70)
print(training_script)

# Save the training script
with open('/Users/shyam.mukherjee/Desktop/research_and_exploration/os_ft/notebooks/train_with_deepspeed.py', 'w') as f:
    f.write(training_script)
print("\nSaved: train_with_deepspeed.py")

## 6. Scaling from 1 to 8 GPUs

Let's analyze how ZeRO scales across different GPU counts:

In [ ]:
def analyze_scaling():
    """
    Analyze memory and throughput scaling with ZeRO.
    """
    model_sizes = {
        'Small (1.3B)': 1_300_000_000,
        'Medium (6.7B)': 6_700_000_000,
        'Large (13B)': 13_000_000_000,
    }
    
    gpu_counts = [1, 2, 4, 8]
    
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    
    for idx, (model_name, num_params) in enumerate(model_sizes.items()):
        ax = axes[idx]
        
        # Calculate memory for each stage
        baseline_mem = []
        stage1_mem = []
        stage2_mem = []
        stage3_mem = []
        
        for num_gpus in gpu_counts:
            baseline_mem.append(calculate_zero_memory(num_params, num_gpus, 0)['total_per_gpu'])
            stage1_mem.append(calculate_zero_memory(num_params, num_gpus, 1)['total_per_gpu'])
            stage2_mem.append(calculate_zero_memory(num_params, num_gpus, 2)['total_per_gpu'])
            stage3_mem.append(calculate_zero_memory(num_params, num_gpus, 3)['total_per_gpu'])
        
        ax.plot(gpu_counts, baseline_mem, 'o-', label='Baseline', linewidth=2, markersize=8)
        ax.plot(gpu_counts, stage1_mem, 's-', label='Stage 1', linewidth=2, markersize=8)
        ax.plot(gpu_counts, stage2_mem, '^-', label='Stage 2', linewidth=2, markersize=8)
        ax.plot(gpu_counts, stage3_mem, 'd-', label='Stage 3', linewidth=2, markersize=8)
        
        # Add GPU memory lines
        ax.axhline(y=40, color='red', linestyle='--', linewidth=2, label='A100 40GB', alpha=0.7)
        ax.axhline(y=80, color='orange', linestyle='--', linewidth=2, label='A100 80GB', alpha=0.7)
        
        ax.set_xlabel('Number of GPUs', fontsize=11)
        ax.set_ylabel('Memory per GPU (GB)', fontsize=11)
        ax.set_title(model_name, fontsize=12, fontweight='bold')
        ax.legend(fontsize=9)
        ax.grid(alpha=0.3)
        ax.set_xticks(gpu_counts)
        ax.set_yscale('log')
    
    plt.tight_layout()
    plt.savefig('/Users/shyam.mukherjee/Desktop/research_and_exploration/os_ft/notebooks/zero_scaling.png',
                dpi=150, bbox_inches='tight')
    plt.show()

analyze_scaling()

# Create detailed scaling table
def create_scaling_table():
    model_name = '6.7B'
    num_params = 6_700_000_000
    gpu_counts = [1, 2, 4, 8]
    
    data = []
    for num_gpus in gpu_counts:
        for stage in [0, 1, 2, 3]:
            mem = calculate_zero_memory(num_params, num_gpus, stage)
            stage_name = f"Stage {stage}" if stage > 0 else "Baseline"
            fits_40gb = "Yes" if mem['total_per_gpu'] <= 40 else "No"
            fits_80gb = "Yes" if mem['total_per_gpu'] <= 80 else "No"
            
            data.append({
                'GPUs': num_gpus,
                'Stage': stage_name,
                'Memory/GPU (GB)': f"{mem['total_per_gpu']:.1f}",
                'Fits A100 40GB': fits_40gb,
                'Fits A100 80GB': fits_80gb,
            })
    
    df = pd.DataFrame(data)
    print(f"\nScaling Analysis for {model_name} Model:")
    print("="*80)
    print(df.to_string(index=False))
    print("="*80)

create_scaling_table()

## 7. When to Use Each ZeRO Stage

### Decision Tree:

```
Does your model fit on 1 GPU with standard training?
├─ Yes → Use standard training or DDP
└─ No
   ├─ Does it fit with ZeRO Stage 1?
   │  ├─ Yes → Use Stage 1 (best speed, minimal overhead)
   │  └─ No
   │     ├─ Does it fit with ZeRO Stage 2?
   │     │  ├─ Yes → Use Stage 2 (good balance)
   │     │  └─ No
   │     │     └─ Use Stage 3 (maximum memory efficiency)
   │     │        ├─ Still doesn't fit?
   │     │           └─ Enable CPU offloading or use more GPUs
```

In [ ]:
# Stage recommendation function
def recommend_zero_stage(num_parameters: int, num_gpus: int, gpu_memory_gb: float = 40):
    """
    Recommend the best ZeRO stage for given configuration.
    
    Args:
        num_parameters: Number of model parameters
        num_gpus: Number of available GPUs
        gpu_memory_gb: Memory per GPU in GB
    
    Returns:
        Recommended stage and reasoning
    """
    # Check each stage
    for stage in [0, 1, 2, 3]:
        mem = calculate_zero_memory(num_parameters, num_gpus, stage)
        # Reserve 20% for activations and overhead
        available_memory = gpu_memory_gb * 0.8
        
        if mem['total_per_gpu'] <= available_memory:
            stage_name = "Baseline (Data Parallel)" if stage == 0 else f"ZeRO Stage {stage}"
            reasoning = f"Memory required: {mem['total_per_gpu']:.1f} GB <= {available_memory:.1f} GB available"
            
            if stage == 0:
                reasoning += "\nNo memory optimization needed."
            elif stage == 1:
                reasoning += "\nOptimizer state partitioning is sufficient."
            elif stage == 2:
                reasoning += "\nGradient partitioning needed. Good balance of speed and memory."
            elif stage == 3:
                reasoning += "\nFull partitioning needed. Maximum memory efficiency."
            
            return stage_name, reasoning
    
    # If nothing fits, recommend Stage 3 with CPU offloading
    return "ZeRO Stage 3 + CPU Offloading", "Model is too large. Use CPU offloading or add more GPUs."

# Test recommendations
test_configs = [
    (1_300_000_000, 1, 40, "GPT-2 (1.3B) on 1x A100 40GB"),
    (1_300_000_000, 4, 40, "GPT-2 (1.3B) on 4x A100 40GB"),
    (6_700_000_000, 1, 40, "GPT-3 (6.7B) on 1x A100 40GB"),
    (6_700_000_000, 4, 40, "GPT-3 (6.7B) on 4x A100 40GB"),
    (13_000_000_000, 8, 40, "GPT-3 (13B) on 8x A100 40GB"),
    (13_000_000_000, 8, 80, "GPT-3 (13B) on 8x A100 80GB"),
]

print("\nZeRO Stage Recommendations:")
print("="*80)
for num_params, num_gpus, gpu_mem, description in test_configs:
    stage, reasoning = recommend_zero_stage(num_params, num_gpus, gpu_mem)
    print(f"\n{description}")
    print(f"Recommendation: {stage}")
    print(f"Reasoning: {reasoning}")
    print("-"*80)

## 8. Advanced Features

### CPU Offloading
- Move optimizer states and/or parameters to CPU
- Trades compute for memory
- Essential for very large models

### ZeRO-Infinity
- Extends ZeRO-3 with NVMe offloading
- Can train trillion-parameter models
- Uses SSD storage when CPU RAM is insufficient

### ZeRO++
- Introduced in 2023
- Reduces communication overhead in Stage 3
- Quantized weights for communication
- Hierarchical partitioning
- ~2x speedup over ZeRO-3

In [ ]:
# Comparison of ZeRO variants
comparison_data = {
    'Feature': [
        'Max Model Size',
        'Memory Efficiency',
        'Speed',
        'Communication',
        'CPU Offload',
        'NVMe Offload',
        'Best For',
    ],
    'Stage 1': [
        'Small (<2B)',
        '4x optimizer',
        'Fastest',
        'Minimal',
        'No',
        'No',
        'Small models, fast training',
    ],
    'Stage 2': [
        'Medium (<10B)',
        '8x total',
        'Fast',
        'Moderate',
        'Optional',
        'No',
        'Most common use case',
    ],
    'Stage 3': [
        'Large (<100B)',
        'Linear with GPUs',
        'Moderate',
        'High',
        'Recommended',
        'No',
        'Large models, limited GPUs',
    ],
    'ZeRO-Infinity': [
        'Huge (1T+)',
        'Extreme',
        'Slower',
        'Very High',
        'Yes',
        'Yes',
        'Trillion-parameter models',
    ],
    'ZeRO++': [
        'Large (<100B)',
        'Same as Stage 3',
        'Fast (2x vs Stage 3)',
        'Reduced',
        'Optional',
        'No',
        'Stage 3 with better speed',
    ],
}

df = pd.DataFrame(comparison_data)
print("\nZeRO Variants Comparison:")
print("="*120)
print(df.to_string(index=False))
print("="*120)

## Summary

**Key Takeaways**:

1. **Problem**: Data parallel training wastes 7/8 of memory with 8 GPUs due to redundancy

2. **Solution**: ZeRO partitions optimizer states, gradients, and parameters across GPUs

3. **Three Stages**:
   - Stage 1: Partition optimizer states (4x memory saving)
   - Stage 2: + Partition gradients (8x memory saving)
   - Stage 3: + Partition parameters (linear scaling with GPUs)

4. **Trade-offs**:
   - Stage 1: Fastest, minimal memory saving
   - Stage 2: Best balance for most use cases
   - Stage 3: Maximum memory efficiency, more communication

5. **When to Use**:
   - Stage 1: Model almost fits, need small optimization
   - Stage 2: Medium models (1-10B), most common
   - Stage 3: Large models (10B+), limited GPU memory

6. **Advanced**:
   - CPU offloading: Trade compute for memory
   - ZeRO-Infinity: Add NVMe for trillion-parameter models
   - ZeRO++: Faster Stage 3 with reduced communication

**Papers**:
- ZeRO: [ZeRO: Memory Optimizations Toward Training Trillion Parameter Models](https://arxiv.org/abs/1910.02054) (Rajbhandari et al., 2020)
- ZeRO-Infinity: [ZeRO-Infinity: Breaking the GPU Memory Wall for Extreme Scale Deep Learning](https://arxiv.org/abs/2104.07857) (Rajbhandari et al., 2021)
- ZeRO++: [ZeRO++: Extremely Efficient Collective Communication for Giant Model Training](https://arxiv.org/abs/2306.10209) (Wang et al., 2023)